# Corrective RAG

[**Corrective Retrieval Augmented Generation**](https://arxiv.org/pdf/2401.15884)
- Most conventional RAG approaches indiscriminately incorporate the retrieved documents, regardless of whether these documents
are relevant or not
- in CRAG, A lightweight retrieval evaluator is designed to assess the overall quality of retrieved documents for a query, returning a confidence degree based on which different knowledge retrieval actions can be triggered
- Non-relevant retrievals can be filtered out before response generation 



In [ ]:
import asyncio
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langgraph.graph import END, START, StateGraph
from loguru import logger
from pydantic import BaseModel, Field

from chain_reaction.config import APIKeys
from chain_reaction.vector_store import load_vector_store

api_keys = APIKeys()

# Configure local data directory
path_parts = Path.cwd().parts
root_dir_index = path_parts.index("chain-reaction")
root_dir = Path(*path_parts[: root_dir_index + 1])
data_dir = root_dir / "data"
data_dir.mkdir(exist_ok=True)

## Load vector store

In [ ]:
vector_store = load_vector_store(
    collection_name="rag-wiki", persist_directory=data_dir / "chroma_db", api_key=api_keys.openai
)

# Number of documents in collection
num_documents = vector_store._client.get_collection("rag-wiki").count()
print(f"# Document chunks: {num_documents:>6,}")

## RAG Functions

### Retrieval

In [ ]:
def retrieve_documents(query: str, *, k: int = 5, fetch_k_mult: int = 10) -> list[Document]:
    """Retrieve a list of relevant documents from the document store based on similarity to query string.

    Args:
        query (str): Query string to search for similar documents.
        k (int): Number of documents to retrieve. Defaults to 5.
        fetch_k_mult (int): Multiple of `k` for initial retrieval. Defaults to 10.

    Returns:
        list[Documents]: List of documents similar to query, ordered by relevance.
    """
    logger.info("Search for {k} documents similar to query: {query}", query=query, k=k)

    # Define a document retriever from the vector store
    retriever = vector_store.as_retriever(
        search_type="mmr",  # Maximal Marginal Relevance (optimizes for relevance and diversity)
        search_kwargs={
            "k": k,
            "fetch_k": k * fetch_k_mult,  # cast a wide net first
            "lambda_mult": 0.5,  # balance relevance vs diversity
        },
    )

    documents = retriever.invoke(query)
    logger.info("{n} documents retrieved", n=len(documents))

    return documents

### Relevance Scoring

In [ ]:
# Relevance score response model
class RelevanceScore(BaseModel):
    """Document relevance score."""

    value: float = Field(
        description=(
            "Value between 0 and 1 for how relevant a document's content is for answering a query."
            " 0.0 = Not relevant."
            " 0.2 = Minimal relevance."
            " 0.5 = Partially relevant."
            " 0.8 = High relevance."
            " 1.0 = Extremely relevant."
        ),
        ge=0,
        le=1,
    )
    reason: str = Field(description="Reason for assigned score.")


# LLM judge
relevance_judge = init_chat_model(model="claude-sonnet-4-6", api_key=api_keys.anthropic).with_structured_output(
    RelevanceScore
)

# Judge prompt
prompt_template = PromptTemplate.from_template(
    """You are a highly specialized question and document review assistant.

    Determine if the following document is helpful for answering this question: {query}

    Document Content
    ----------------------------------------------------------------------------
    Title: {title}
    Content: {content}
    ----------------------------------------------------------------------------

    Assign a relevance score between 0 and 1 for how relevant the document's content is for answering the question.
    Also provide a reason for why you assigned the score.
    """
)

# Judge chain
relevance_chain = prompt_template | relevance_judge


async def async_score_document(query: str, document: Document) -> RelevanceScore:
    """Determine if a document is relevant for answering a query using LLM as a judge."""
    response: RelevanceScore = await relevance_chain.ainvoke({
        "query": query,
        "title": document.metadata.get("title", ""),
        "content": document.page_content,
    })
    logger.info(
        "Doc {doc_id} relevance score {score}",
        doc_id=document.id,
        score=response.value,
    )
    return response


async def async_score_documents(query: str, documents: list[Document]) -> list[RelevanceScore]:
    """Score retrieved documents for relevance to a query using LLM as a judge.

    Args:
        query (str): The same query string used in retrieve_documents.
        documents (list[Document]): Documents that might be relevant to the query.

    Returns:
        list[RelevanceScore]: Relevance score for each document.
    """
    return await asyncio.gather(*[async_score_document(query=query, document=doc) for doc in documents])

### Answer

In [ ]:
class Answer(BaseModel):
    """Documented grounded answer."""

    value: str = Field(description="Answer to query grounded in information only from documents.")
    citations: list[str] = Field(description="Document IDs used in answer", default_factory=list)
    confidence: float = Field(
        description="Confidence of answer based on relevance information consistency of documents used.", ge=0, le=1
    )


answer_model = init_chat_model(model="claude-sonnet-4-6", api_key=api_keys.anthropic).with_structured_output(Answer)

answer_prompt_template = PromptTemplate.from_template(
    """You are a helpful question answering assistant.

    Instructions:
    - Answer the question as concisely as possible.
    - Use only the information from the provided documents to answer the question.
    - Give more weight to information from documents with higher relevance scores.
    - Never make information up.
    - If insufficient context is provided by the document to answer, respond:
        "I cannot answer the question with the provided information."
    - Cite the document IDs used to answer the question.
    - Qualify the quality of your answer based on the relevance of the documents used.
        (The higher the relevance the stronger your answer confidence.)

    Question: {query}

    Document Information:
    ----------------------------------------------------------------------------
    {context}
    """
)


def generate_answer(
    query: str,
    documents: list[Document],
    relevance_scores: list[RelevanceScore],
    answer_prompt_template: PromptTemplate,
    relevance_th: float,
) -> Answer | None:
    """Generate an answer to query using only context from relevant documents.

    Args:
        query (str): Question to answer.
        documents (list[Document]): Retrieved documents to answer question.
        relevance_scores (list[RelevanceScore]): Relevance scores for documents.
        answer_prompt_template (PromptTemplate): Prompt template to combine instructions, query, context.
        relevance_th (float): Minimum relevance threshold to use a document.

    Returns:
        Answer | None: Answer to question or None if all documents were filtered out.

    Raises:
        ValueError: If no documents were provided.
    """
    if not documents:
        msg = "No documents provided to answer question."
        logger.warning(msg)
        raise ValueError(msg)

    # Zip docs and relevance scores together
    docs_n_scores: list[tuple[Document, RelevanceScore]] = list(zip(documents, relevance_scores, strict=True))

    # Filter out documents below relevance threshold
    docs_n_scores = [(doc, score) for doc, score in docs_n_scores if score.value >= relevance_th]
    logger.info(
        "{num_filtered} documents filtered out at th: {th}",
        num_filtered=len(documents) - len(docs_n_scores),
        th=relevance_th,
    )
    if not docs_n_scores:
        logger.info(
            "No relevant documents left to answer question. Max relevance: {max_rel}",
            max_rel=max(relevance_scores, key=lambda s: s.value),
        )
        return None

    # Sort remaining documents by relevance (highest to lowest)
    docs_n_scores = sorted(docs_n_scores, key=lambda x: x[1].value, reverse=True)

    # Compile context from documents
    context = "".join(_format_document(doc, score) for doc, score in docs_n_scores)

    # Invoke answer model
    answer = (answer_prompt_template | answer_model).invoke({"query": query, "context": context})

    return answer


def _format_document(document: Document, relevance: RelevanceScore) -> str:
    """Format a document and it's metadata as a string."""
    return f"""
    Document ID: {document.id}
    Relevance: {relevance.value}
    Relevance Reason: {relevance.reason}
    Document Title: {document.metadata.get("title", "")}
    Document Content: {document.page_content}
    ----------------------------------------------------------------------------
    """

### Test functions

In [ ]:
query = "What role do embedding models play in RAG?"
documents = retrieve_documents(query, k=10)
relevance_scores = await async_score_documents(query, documents)
generate_answer(
    query=query,
    documents=documents,
    relevance_scores=relevance_scores,
    answer_prompt_template=answer_prompt_template,
    relevance_th=0.2,
)

## RAG Graph State

In [ ]:
class RAGState(BaseModel):
    """State of RAG Graph.

    Attributes:
        query (str): Query.
        answer (Answer): Generated answer to query.
        documents (list[Document]): List of retrieved documents.
        relevance_scores (list[Relevance]): List of relevance scores.
        answer_prompt_template (PromptTemplate): Prompt template to combine instructions, query, context.
        k (int): Number of documents to retrieve. Defaults to 10.
        relevance_th (float): Minimum relevance threshold to use a document. Defaults to 0.2.
    """

    query: str
    answer: Answer | None = None
    documents: list[Document] = Field(default_factory=list)
    relevance_scores: list[RelevanceScore] = Field(default_factory=list)
    answer_prompt_template: PromptTemplate = answer_prompt_template
    k: int = 10
    relevance_th: float = 0.20

## RAG Nodes

In [ ]:
def retrieve(state: RAGState) -> RAGState:
    """Document retrieval node."""
    logger.info("--retrieve node--")
    documents = retrieve_documents(query=state.query, k=state.k)
    return state.model_copy(update={"documents": documents})


async def grade(state: RAGState) -> RAGState:
    """Document grading node."""
    logger.info("--grade node--")
    relevance_scores = await async_score_documents(query=state.query, documents=state.documents)
    return state.model_copy(update={"relevance_scores": relevance_scores})


def answer(state: RAGState) -> RAGState:
    """Query answering node."""
    logger.info("--answer node--")
    answer = generate_answer(
        query=state.query,
        documents=state.documents,
        relevance_scores=state.relevance_scores,
        answer_prompt_template=state.answer_prompt_template,
        relevance_th=state.relevance_th,
    )
    return state.model_copy(update={"answer": answer})

## RAG Graph

In [ ]:
graph = (
    StateGraph(RAGState)
    .add_sequence([retrieve, grade, answer])
    .add_edge(START, "retrieve")
    .add_edge("retrieve", "grade")
    .add_edge("grade", "answer")
    .add_edge("grade", END)
    .compile()
)
graph

In [ ]:
output = await graph.ainvoke({"query": "When was RAG developed?"})
output = RAGState(**output)
output.answer

In [ ]:
output = await graph.ainvoke({"query": "How do you bake RAG bread?"})
output = RAGState(**output)
output.answer